# Minimap Dot Detection — De-risk Spike (Phase 0)

**目的**: 10 個のミニマップドット (味方 5・敵 5) を YOLO 再学習で安定識別できるかを 1 日で判定する。

**判定基準**:
- `enemy_dot` の mAP@50 ≥ 0.7 → 本学習で進める（Phase 4 のスコープを「10 人個別トラッキング」で確定）
- それ未満 → HSV 色バケット + ハンガリアン法フォールバックを Phase 4 のベースラインに採用

**追跡 Issue**: tsukasakaneko/tauriAICoaching#9

**Notebook 構成**:
1. セットアップ
2. 既存録画からミニマップクロップを 2fps で 200 枚抽出
3. 手動ラベル (CVAT/Roboflow へエクスポート)
4. YOLOv8n の fine-tune (Colab GPU 推奨)
5. クラス別 mAP/Precision/Recall を測定
6. HSV 色バケット + ハンガリアン法フォールバックの実装スケッチ
7. 決定セクション (Phase 4 への申し送り)

**Notebook の使い方**: 上から順に実行。各セクションの TODO を埋めながら進める。

## 1. セットアップ

In [ ]:
# TODO: 必要に応じて pip install
# !pip install ultralytics opencv-python pillow numpy scipy

import os
import json
from pathlib import Path

# パス設定 (リポジトリルートから notebook を実行する想定)
REPO_ROOT = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd().parent
RECORDINGS_DIR = REPO_ROOT / 'recordings'  # ライブキャプチャの出力先 (要確認)
DATASET_DIR = Path.home() / 'datasets' / 'minimap_dot_spike'
DATASET_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CROP_COUNT = 200
TARGET_FPS = 2

print(f'Repo root: {REPO_ROOT}')
print(f'Recordings: {RECORDINGS_DIR}')
print(f'Dataset: {DATASET_DIR}')

## 2. ミニマップクロップ抽出

既存録画 (`recordings/*.mp4`) から 2fps でフレームを取り、ミニマップ領域だけクロップして 200 枚保存する。

**TODO**:
- ミニマップ領域の座標を確認 (`backend/services/yoloInference.js` の minimap モデルが期待する入力範囲を参照)
- ffmpeg で抽出 → PIL でクロップ → `DATASET_DIR/raw/` に保存
- 複数試合・複数マップから取り、偏りを避ける

In [ ]:
# TODO: クロップ抽出実装
# 例:
# import subprocess, cv2
# MINIMAP_BBOX = (x, y, w, h)  # 要キャリブレーション
# for video in RECORDINGS_DIR.glob('*.mp4'):
#     ...
raise NotImplementedError('クロップ抽出をここに実装する')

## 3. 手動ラベル

`DATASET_DIR/raw/` の 200 枚を CVAT または Roboflow にアップロードし、以下のクラスで box ラベリング:

- `local_player` — 自分 (中央の白/大きいドット)
- `ally_dot` — 味方 4 人 (緑系)
- `enemy_dot` — 敵 5 人 (赤系、見えるときのみ)
- `dead_marker` — 死体マーカー (X 印)

YOLO 形式でエクスポートして `DATASET_DIR/yolo/{images,labels}/{train,val}/` に配置。

**目標分布**: train 160 / val 40。クラス別出現数はラベル後にここで確認。

In [ ]:
# TODO: ラベル分布の確認
# from collections import Counter
# labels = []
# for txt in (DATASET_DIR / 'yolo' / 'labels').rglob('*.txt'):
#     labels.extend(int(line.split()[0]) for line in txt.read_text().splitlines())
# print(Counter(labels))
raise NotImplementedError('ラベル分布チェックを実装する')

## 4. YOLOv8n の fine-tune

Colab GPU で実行推奨 (ローカル CPU でも可だが遅い)。

In [ ]:
# TODO: 学習実行
# from ultralytics import YOLO
# model = YOLO('yolov8n.pt')
# results = model.train(
#     data=str(DATASET_DIR / 'yolo' / 'data.yaml'),
#     epochs=100,
#     imgsz=640,
#     batch=16,
#     name='minimap_dot_spike',
# )
# print(results)
raise NotImplementedError('学習をここで実行する')

## 5. mAP / Precision / Recall を測定

クラス別の mAP@50, P, R を出して `enemy_dot` を判定基準と照らす。

In [ ]:
# TODO: 評価
# metrics = model.val()
# print('mAP per class:')
# for cls_idx, name in enumerate(['local_player', 'ally_dot', 'enemy_dot', 'dead_marker']):
#     print(f'  {name}: mAP50={metrics.box.map50_per_class[cls_idx]:.3f}')
raise NotImplementedError('評価結果を出力する')

## 6. HSV 色バケット + ハンガリアン法フォールバックの実装スケッチ

YOLO がダメだった場合の代替案。各ドットを HSV ヒストグラムで `team_red` / `team_green` / `self_white` に分け、フレーム間で位置距離 + 色類似度の Hungarian assignment でトラッキング。

個別 ally の再識別は諦め、「5 ally の連続軌跡」「敵は見えた瞬間だけ点」を提供する。

In [ ]:
# TODO: フォールバック実装スケッチ
# - HSV マスク → connected components → 各成分の重心
# - 色相判定で team_red / team_green / self_white に分類
# - 前フレームの track と Hungarian assignment (scipy.optimize.linear_sum_assignment)
raise NotImplementedError('フォールバックを実装する')

## 7. 決定セクション

(本ノートブックを実行した後、ここに結果と Phase 4 への申し送りを記入する)

### データセット
- 抽出枚数: TBD
- ラベル分布: TBD

### YOLO 結果
- local_player mAP50: TBD
- ally_dot     mAP50: TBD
- enemy_dot    mAP50: TBD
- dead_marker  mAP50: TBD

### 採用する手法
- [ ] YOLO 本学習 (`enemy_dot` mAP ≥ 0.7 の場合)
- [ ] HSV + Hungarian フォールバック (上記未達の場合)

### Phase 4 (#13) への申し送り
- TBD